In [15]:
import arcpy
import os

arcpy.env.overwriteOutput = True

out_gdb = r'C:\Users\colet\Documents\GEOG5890_Capstone\GEOG5890_Capstone_GDB\GeoAI_Capstone.gdb'

crash_grid = os.path.join(out_gdb, 'Crash_Grid')
landuse = r'C:\Users\colet\Documents\GEOG5890_Capstone\R_Land_Use\Land_Use_2020\Land_Use_Grouped.shp'

landuse_proj = os.path.join(out_gdb, 'Land_Use_Grouped_proj')

desc = arcpy.Describe(crash_grid)
crs = desc.spatialReference

arcpy.management.Project(landuse, landuse_proj, crs)

<Result 'C:\\Users\\colet\\Documents\\GEOG5890_Capstone\\GEOG5890_Capstone_GDB\\GeoAI_Capstone.gdb\\Land_Use_Grouped_proj'>

In [16]:
landuse = landuse_proj

grid_id = 'Grid_ID'
area_field = 'Area_m2'
landuse_field = 'LU_Group'

landuse_types = ['Residential', 'Commercial_Office', 'Industrial_Transport', 'Mixed_Use',
                 'Public_Cultural_Rec', 'Parking', 'Vacant_Agriculture', 'Other_Unknown']

In [17]:
landuse_fixed = os.path.join(out_gdb, 'LandUse_Grouped_Fixed')
arcpy.management.CopyFeatures(landuse, landuse_fixed)
arcpy.management.RepairGeometry(landuse_fixed, 'DELETE_NULL')

<Result 'C:\\Users\\colet\\Documents\\GEOG5890_Capstone\\GEOG5890_Capstone_GDB\\GeoAI_Capstone.gdb\\LandUse_Grouped_Fixed'>

In [18]:
for landuse_type in landuse_types:
    
    landuse = landuse_fixed
    
    percent_field = 'LU_' + landuse_type + '_pct'

    intersect_fc = os.path.join(out_gdb, 'temp_intersect')
    summary_table = os.path.join(out_gdb, 'temp_summary')

    arcpy.management.MakeFeatureLayer(landuse, 'landuse_layer')

    where_clause = (
        f'{arcpy.AddFieldDelimiters(landuse, landuse_field)} '
        f"= '{landuse_type}'")

    arcpy.management.SelectLayerByAttribute('landuse_layer', 'NEW_SELECTION', where_clause)

    arcpy.analysis.Intersect([crash_grid, 'landuse_layer'], intersect_fc, 'ALL', None, 'INPUT')

    arcpy.management.AddField(intersect_fc, 'Fragment_m2', 'DOUBLE')
    arcpy.management.CalculateGeometryAttributes(intersect_fc, [['Fragment_m2', 'AREA']],
                                                 area_unit = 'SQUARE_METERS')

    arcpy.analysis.Statistics(intersect_fc, summary_table, [['Fragment_m2', 'SUM']], [grid_id])

    existing_fields = [f.name for f in arcpy.ListFields(crash_grid)]

    if percent_field not in existing_fields:
        arcpy.management.AddField(crash_grid, percent_field, 'DOUBLE')

    arcpy.management.JoinField(crash_grid, grid_id, summary_table, grid_id, ['SUM_Fragment_m2'])

    expression = (
        '0 if !SUM_Fragment_m2! == None '
        'else (!SUM_Fragment_m2! / !Area_m2!) * 100'
    )

    arcpy.management.CalculateField(crash_grid, percent_field, expression, 'PYTHON3')

    arcpy.management.DeleteField(crash_grid, 'SUM_Fragment_m2')

    if arcpy.Exists(intersect_fc):
        arcpy.management.Delete(intersect_fc)

    if arcpy.Exists(summary_table):
        arcpy.management.Delete(summary_table)

    arcpy.management.Delete('landuse_layer')

In [14]:
unclassified_field = 'LU_Unclassified_pct'
existing_fields = [f.name for f in arcpy.ListFields(crash_grid)]

arcpy.management.AddField(crash_grid, unclassified_field, 'DOUBLE')

percentage_fields = ['LU_Residential_pct', 'LU_Commercial_Office_pct', 'LU_Industrial_Transport_pct',
                     'LU_Mixed_Use_pct', 'LU_Public_Cultural_Rec_pct', 'LU_Parking_pct', 'LU_Vacant_Agriculture_pct',
                     'LU_Other_Unknown_pct']

expression = '100 - (' + ' + '.join([f'!{field}!' for field in percentage_fields]) + ')'

arcpy.management.CalculateField(crash_grid, unclassified_field, expression, 'PYTHON3')

<Result 'C:\\Users\\colet\\Documents\\GEOG5890_Capstone\\GEOG5890_Capstone_GDB\\GeoAI_Capstone.gdb\\Crash_Grid'>